In [3]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
logger = logging.getLogger(__name__)

# ==============================
# KONEKSI POSTGRESQL
# ==============================
conn = psycopg2.connect(
    host="postgres",
    database="olist_ecommerce",
    user="airflow",
    password="airflow",
    port="5432"
)
cur = conn.cursor()
logger.info("✅ Koneksi PostgreSQL berhasil!")

try:
    # ==============================
    # BUAT TABEL (hanya sekali, skip kalau sudah ada)
    # ==============================
    cur.execute("""
        CREATE TABLE IF NOT EXISTS raw.customers (
            customer_id         VARCHAR PRIMARY KEY,
            customer_unique_id  VARCHAR,
            customer_zip_code   VARCHAR,
            customer_city       VARCHAR,
            customer_state      VARCHAR,
            created_at          TIMESTAMP DEFAULT NOW()
        );

        CREATE TABLE IF NOT EXISTS raw.products (
            product_id                  VARCHAR PRIMARY KEY,
            product_category_name       VARCHAR,
            product_name_length         FLOAT,
            product_description_length  FLOAT,
            product_photos_qty          FLOAT,
            product_weight_g            FLOAT,
            product_length_cm           FLOAT,
            product_height_cm           FLOAT,
            product_width_cm            FLOAT,
            created_at                  TIMESTAMP DEFAULT NOW()
        );

        CREATE TABLE IF NOT EXISTS raw.sellers (
            seller_id        VARCHAR PRIMARY KEY,
            seller_zip_code  VARCHAR,
            seller_city      VARCHAR,
            seller_state     VARCHAR,
            created_at       TIMESTAMP DEFAULT NOW()
        );
    """)
    conn.commit()
    logger.info("✅ Tabel berhasil dibuat!")

    # ==============================
    # LOAD CUSTOMERS
    # ==============================
    df_customers = pd.read_csv('/home/jovyan/data/olist_customers_dataset.csv')
    logger.info(f"📂 Customers: {len(df_customers)} rows")

    execute_batch(cur, """
        INSERT INTO raw.customers (
            customer_id, customer_unique_id, customer_zip_code,
            customer_city, customer_state
        )
        VALUES (%s,%s,%s,%s,%s)
        ON CONFLICT (customer_id) DO NOTHING;
    """, [
        (
            row["customer_id"],
            row["customer_unique_id"],
            row["customer_zip_code_prefix"],
            row["customer_city"],
            row["customer_state"]
        )
        for _, row in df_customers.iterrows()
    ])
    conn.commit()
    logger.info(f"✅ Customers loaded: {len(df_customers)} rows")

    # ==============================
    # LOAD PRODUCTS
    # ==============================
    df_products = pd.read_csv('/home/jovyan/data/olist_products_dataset.csv')

    # Perbaiki typo nama kolom dari CSV
    df_products = df_products.rename(columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length"
    })

    # Handle NULL dan konversi tipe data
    for col in ["product_name_length", "product_description_length",
                "product_photos_qty", "product_weight_g",
                "product_length_cm", "product_height_cm", "product_width_cm"]:
        df_products[col] = pd.to_numeric(df_products[col], errors="coerce")

    # Ganti NaN jadi None agar masuk sebagai NULL di PostgreSQL
    df_products = df_products.where(pd.notnull(df_products), None)
    logger.info(f"📂 Products: {len(df_products)} rows")

    execute_batch(cur, """
        INSERT INTO raw.products (
            product_id, product_category_name,
            product_name_length, product_description_length,
            product_photos_qty, product_weight_g,
            product_length_cm, product_height_cm, product_width_cm
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
        ON CONFLICT (product_id) DO NOTHING;
    """, [
        (
            row["product_id"],
            row["product_category_name"],
            row["product_name_length"],
            row["product_description_length"],
            row["product_photos_qty"],
            row["product_weight_g"],
            row["product_length_cm"],
            row["product_height_cm"],
            row["product_width_cm"]
        )
        for _, row in df_products.iterrows()
    ])
    conn.commit()
    logger.info(f"✅ Products loaded: {len(df_products)} rows")

    # ==============================
    # LOAD SELLERS
    # ==============================
    df_sellers = pd.read_csv('/home/jovyan/data/olist_sellers_dataset.csv')
    logger.info(f"📂 Sellers: {len(df_sellers)} rows")

    execute_batch(cur, """
        INSERT INTO raw.sellers (
            seller_id, seller_zip_code,
            seller_city, seller_state
        )
        VALUES (%s,%s,%s,%s)
        ON CONFLICT (seller_id) DO NOTHING;
    """, [
        (
            row["seller_id"],
            row["seller_zip_code_prefix"],
            row["seller_city"],
            row["seller_state"]
        )
        for _, row in df_sellers.iterrows()
    ])
    conn.commit()
    logger.info(f"✅ Sellers loaded: {len(df_sellers)} rows")

except Exception as e:
    conn.rollback()
    logger.error(f"❌ Pipeline error: {e}")
    raise

finally:
    cur.close()
    conn.close()
    logger.info("🏁 Semua master data berhasil diload ke PostgreSQL!")

2026-03-28 15:36:45,989 - ✅ Koneksi PostgreSQL berhasil!
2026-03-28 15:36:45,991 - ✅ Tabel berhasil dibuat!
2026-03-28 15:36:46,236 - 📂 Customers: 99441 rows
2026-03-28 15:36:50,560 - ✅ Customers loaded: 99441 rows
2026-03-28 15:36:50,611 - 📂 Products: 32951 rows
2026-03-28 15:36:52,470 - ✅ Products loaded: 32951 rows
2026-03-28 15:36:52,477 - 📂 Sellers: 3095 rows
2026-03-28 15:36:52,610 - ✅ Sellers loaded: 3095 rows
2026-03-28 15:36:52,610 - 🏁 Semua master data berhasil diload ke PostgreSQL!
